# Diamond Pipeline — Load CSV → Download IGI PDFs (15 threads) → Save Enriched CSV

1. **Step 1** — Load `luvansh_updated.csv` (2,064 diamonds with SKU + certificate URLs)
2. **Step 2** — Download each IGI PDF with 15 threads → extract proportions (crown angle, pavilion angle, etc.)
3. **Step 3** — Merge original data + PDF data → save `diamonds_full.csv`

In [ ]:
!pip install -q pdfplumber requests pandas
print("Dependencies installed.")

## 1. Load CSV

In [ ]:
import re, time, io, os
import requests
import pdfplumber
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

THREADS = 15
INPUT_CSV = "luvansh_updated.csv"
OUTPUT_CSV = "diamonds_full.csv"

# Find the CSV file
csv_path = None
for p in [f"/content/{INPUT_CSV}", INPUT_CSV, os.path.join(os.getcwd(), INPUT_CSV)]:
    if os.path.exists(p):
        csv_path = p
        break

if csv_path is None:
    raise FileNotFoundError(f"'{INPUT_CSV}' not found — upload it to Colab.")

df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} diamonds from {csv_path}")
print(f"Columns: {list(df.columns)}")
print(f"\nDiamonds with certificate URL: {df['web_certificate_url'].notna().sum()}")
df.head(5)

## 2. Download IGI PDFs (15 Threads) → Extract Proportions

Each PDF URL: `https://api.igi.org/viewpdf.php?r=LG738516036`

Extracts: crown angle, pavilion angle, crown height %, pavilion depth %, girdle, culet, and verifies table/depth/measurements.

In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}


def extract_from_pdf(pdf_url):
    """
    Download an IGI PDF and extract all proportions with pdfplumber.
    Returns dict with all extracted fields.
    """
    resp = requests.get(pdf_url, headers=HEADERS, timeout=30)
    resp.raise_for_status()

    with pdfplumber.open(io.BytesIO(resp.content)) as pdf:
        full_text = "\n".join(page.extract_text() or "" for page in pdf.pages)

    props = {"pdf_raw_text": full_text[:500]}  # keep first 500 chars for debug

    def find_f(pattern):
        m = re.search(pattern, full_text, re.I)
        return float(m.group(1)) if m else None

    def find_s(pattern):
        m = re.search(pattern, full_text, re.I)
        return m.group(1).strip() if m else None

    # ── Measurements & L/W ────────────────────────────────
    meas_m = re.search(
        r'(\d+\.\d+\s*[-\u2013]\s*\d+\.\d+\s*[Xx\u00d7]\s*\d+\.\d+)', full_text)
    if meas_m:
        props["pdf_measurements"] = meas_m.group(1).strip()
        dims = re.findall(r'(\d+\.\d+)', meas_m.group(1))
        if len(dims) >= 2:
            l, w = float(dims[0]), float(dims[1])
            if min(l, w) > 0:
                props["pdf_lw_ratio"] = round(max(l, w) / min(l, w), 3)

    # ── Named fields ──────────────────────────────────────
    props["pdf_table_pct"]      = find_f(r'Table\s*:?\s*(\d+(?:\.\d+)?)\s*%')
    props["pdf_depth_pct"]      = find_f(r'Depth\s*:?\s*(\d+(?:\.\d+)?)\s*%')
    props["pdf_crown_angle"]    = find_f(r'Crown\s*Angle\s*:?\s*(\d+\.\d+)')
    props["pdf_pavilion_angle"] = find_f(r'Pavilion\s*Angle\s*:?\s*(\d+\.\d+)')
    props["pdf_crown_height"]   = find_f(r'Crown\s*Height\s*:?\s*(\d+\.\d+)')
    props["pdf_pavilion_depth"] = find_f(r'Pavilion\s*Depth\s*:?\s*(\d+\.\d+)')

    # ── Proportions diagram: "13.5% 58% 33.1\u00b0 40.9\u00b0 43% Pointed 61%" ──
    prop_m = re.search(
        r'(\d+\.\d+)%\s+(\d+)%\s+(\d+\.\d+)[\u00b0]\s+(\d+\.\d+)[\u00b0]\s+'
        r'(\d+(?:\.\d+)?)%\s+\w+\s+(\d+(?:\.\d+)?)%',
        full_text)
    if prop_m:
        props.setdefault("pdf_crown_height",   float(prop_m.group(1)))
        props.setdefault("pdf_table_pct",      float(prop_m.group(2)))
        props.setdefault("pdf_crown_angle",    float(prop_m.group(3)))
        props.setdefault("pdf_pavilion_angle", float(prop_m.group(4)))
        props.setdefault("pdf_pavilion_depth", float(prop_m.group(5)))
        props.setdefault("pdf_depth_pct",      float(prop_m.group(6)))

    # ── Two angles side by side ───────────────────────────
    if not props.get("pdf_crown_angle") or not props.get("pdf_pavilion_angle"):
        ang_m = re.search(r'(\d{2}\.\d+)[\u00b0]\s+(\d{2}\.\d+)[\u00b0]', full_text)
        if ang_m:
            props.setdefault("pdf_crown_angle",    float(ang_m.group(1)))
            props.setdefault("pdf_pavilion_angle", float(ang_m.group(2)))

    # ── Grading fields ────────────────────────────────────
    props["pdf_polish"]       = find_s(r'Polish\s*:?\s*(EXCELLENT|VERY\s*GOOD|GOOD|FAIR|POOR)')
    props["pdf_symmetry"]     = find_s(r'Symmetry\s*:?\s*(EXCELLENT|VERY\s*GOOD|GOOD|FAIR|POOR)')
    props["pdf_fluorescence"] = find_s(r'Fluorescence\s*:?\s*(NONE|FAINT|MEDIUM|STRONG|VERY\s*STRONG)')
    props["pdf_girdle"]       = find_s(r'Girdle\s*:?\s*([A-Za-z][A-Za-z\s]*(?:\(Faceted\))?)')
    props["pdf_culet"]        = find_s(r'Culet\s*:?\s*(None|Pointed|Very\s*Small|Small|Medium|Large)')
    props["pdf_cut_grade"]    = find_s(r'Cut\s*(?:Grade)?\s*:?\s*(IDEAL|EXCELLENT|VERY\s*GOOD|GOOD)')
    props["pdf_carat"]        = find_f(r'Carat\s*Weight\s*:?\s*(\d+\.\d+)')
    props["pdf_color"]        = find_s(r'Color\s*Grade\s*:?\s*([A-Z])\b')
    props["pdf_clarity"]      = find_s(r'Clarity\s*Grade\s*:?\s*(FL|IF|VVS[12]|VS[12]|SI[12]|I[123])')

    return props

print("PDF extraction function ready.")

# ── Quick test on first diamond ───────────────────────────
test_url = df["web_certificate_url"].dropna().iloc[0]
print(f"\nTesting: {test_url}")
test_props = extract_from_pdf(test_url)
for k, v in test_props.items():
    if k != "pdf_raw_text" and v is not None:
        print(f"  {k}: {v}")
print(f"\nRaw text preview:\n{test_props.get('pdf_raw_text', '')[:300]}")

In [ ]:
# ── Run 15-threaded PDF extraction ────────────────────────
print_lock = Lock()
progress = {"done": 0}

# Only process rows that have a certificate URL
has_url = df["web_certificate_url"].notna()
indices = df.index[has_url].tolist()
total = len(indices)
print(f"Downloading {total} IGI PDFs with {THREADS} threads ...\n")

# Pre-create result columns
pdf_columns = [
    "pdf_measurements", "pdf_lw_ratio",
    "pdf_table_pct", "pdf_depth_pct",
    "pdf_crown_angle", "pdf_pavilion_angle",
    "pdf_crown_height", "pdf_pavilion_depth",
    "pdf_polish", "pdf_symmetry", "pdf_fluorescence",
    "pdf_girdle", "pdf_culet",
    "pdf_cut_grade", "pdf_carat", "pdf_color", "pdf_clarity",
    "pdf_error",
]
for col in pdf_columns:
    df[col] = None


def process_row(idx):
    """Download one PDF and return (idx, props_dict)."""
    url = df.at[idx, "web_certificate_url"]
    sku = df.at[idx, "web_sku"] if "web_sku" in df.columns else ""
    props = {}
    try:
        props = extract_from_pdf(url)
    except Exception as e:
        props["pdf_error"] = str(e)[:120]

    with print_lock:
        progress["done"] += 1
        d = progress["done"]
        if d <= 3 or d % 100 == 0 or d == total:
            ca = props.get('pdf_crown_angle', '-')
            pa = props.get('pdf_pavilion_angle', '-')
            err = props.get('pdf_error', '')
            if err:
                print(f"  [{d}/{total}] {sku}  ERROR: {err[:60]}")
            else:
                print(f"  [{d}/{total}] {sku}  CrAngle={ca}  PavAngle={pa}")

    return idx, props


start_time = time.time()

with ThreadPoolExecutor(max_workers=THREADS) as executor:
    futures = {executor.submit(process_row, idx): idx for idx in indices}
    for future in as_completed(futures):
        idx, props = future.result()
        for k, v in props.items():
            if k in pdf_columns:
                df.at[idx, k] = v

elapsed = time.time() - start_time

# ── Summary ───────────────────────────────────────────────
got_angle = df["pdf_crown_angle"].notna().sum()
got_error = df["pdf_error"].notna().sum()
print(f"\nDone in {elapsed:.0f}s ({elapsed/total:.2f}s per diamond)")
print(f"Crown angle extracted: {got_angle}/{total}")
print(f"Errors: {got_error}/{total}")

if got_error > 0:
    print(f"\nSample errors:")
    print(df[df["pdf_error"].notna()][["web_sku", "web_certificate_url", "pdf_error"]].head(5).to_string())

## 3. Save Combined CSV

In [ ]:
# Drop the debug raw text column before saving
save_cols = [c for c in df.columns if c != "pdf_raw_text"]
df_out = df[save_cols]

df_out.to_csv(OUTPUT_CSV, index=False)
print(f"Saved → {OUTPUT_CSV}  ({len(df_out)} rows, {len(df_out.columns)} columns)")

# ── Column summary ────────────────────────────────────────
print(f"\n{'Column':<25} {'Non-null':>8}  {'Example'}")
print("─" * 70)
for col in df_out.columns:
    nn = df_out[col].notna().sum()
    ex = df_out[col].dropna().iloc[0] if nn > 0 else ""
    print(f"{col:<25} {nn:>8}  {str(ex)[:40]}")

df_out.head(5)

In [ ]:
# ── Download in Colab ─────────────────────────────────────
try:
    from google.colab import files
    files.download(OUTPUT_CSV)
except ImportError:
    print(f"File saved locally: {OUTPUT_CSV}")